# 🛒 Lab Session: Generate Text with Pre-trained GPT Model
## Use Case: Product Description Writer (E-commerce)

---

### 🎯 What Will You Build?
By the end of this lab, you will have a working Python program that:
- Takes a **product name and its features** as input
- Sends that information to **OpenAI's GPT model** via **LangChain**
- Gets back a **professionally written product description** — ready for an e-commerce website

---

### 🧠 What Do You Already Know?
You have learned:
- **ANN** → Neural Networks that learn patterns from data
- **ML** → Models that make predictions

### 🔁 How Does This Connect?
> GPT is a **pre-trained** neural network — it already learned from billions of sentences on the internet.  
> We are **not training** it. We are simply **sending it a task (prompt)** and using its response.  
> Think of it like hiring an expert writer — you give instructions, they write.

---

### 🏗️ Architecture of What We Are Building

```
Your Python Code
      │
      ▼
  LangChain (the connector)
      │
      ▼
  OpenAI GPT (the AI brain)
      │
      ▼
  Product Description (the output)
```

**LangChain** is a Python library that makes it easy to connect your code with AI models like GPT.  
It handles the communication, formatting, and response — so you don't have to write that plumbing code yourself.


---
## Step 1: Install Required Libraries

### 📦 What Are We Installing?

| Library | Purpose |
|---|---|
| `langchain` | The main connector framework — helps structure prompts and chain AI calls |
| `langchain-openai` | LangChain's specific plugin to talk to OpenAI models |
| `openai` | The official OpenAI Python SDK used internally by LangChain |

### 💡 Why LangChain Instead of Using OpenAI Directly?
You *could* call OpenAI directly. But LangChain gives you:
- **Prompt Templates** — reusable, structured prompts with variables
- **Chains** — connect multiple AI steps together (useful in bigger projects)
- **A consistent pattern** — same code works with GPT, Gemini, Claude, etc.

Run the cell below to install everything needed:


In [1]:
%pip install langchain langchain-openai openai --quiet
%pip install python-dotenv
print("✅ Libraries installed successfully!")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
✅ Libraries installed successfully!



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## Step 2: Import Libraries

### 📖 What Are We Importing and Why?

```
from langchain_openai import ChatOpenAI
```
→ This is the LangChain class that represents the GPT model.  
→ Think of it as your "AI assistant object" that you can talk to.

```
from langchain.prompts import ChatPromptTemplate
```
→ A **Prompt Template** is like a **form letter** — it has fixed text with blank spaces for your variable inputs.  
→ Example: *"Write a description for {product_name} with these features: {features}"*

```
from langchain_core.output_parsers import StrOutputParser
```
→ The model returns a response object. This parser **extracts just the text** from it.  
→ Think of it as opening an envelope and taking out only the letter inside.

```
import os
```
→ Standard Python library to set **environment variables** like your API key — securely.


In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


---
## Step 3: Load Your OpenAI API Key

### 🔑 What Is an API Key?
Think of the **OpenAI API Key** as your **membership card** to access OpenAI's GPT models.  
Without it, your code cannot communicate with GPT.

### 🔐 Secure Approach: Use a  File
Instead of pasting the key directly in the notebook (unsafe!), we store it in a  file  
in the project root and load it with .

**Your  file contains:**


**Why this is safe:**
-  is listed in  — it will never be committed to git
- The key never appears in the notebook itself
- Safe to share the notebook with teammates

**Install once** (already done in Step 1): Requirement already satisfied: python-dotenv in d:\johnpaul\learning\learn-ai\generative-ai-notes\.venv\lib\site-packages (1.2.2)

Run the cell below to load your key:

In [3]:
from dotenv import load_dotenv
import os

# override=True ensures the latest key from .env is always used,
# even if the kernel already has an old key cached in memory
load_dotenv(override=True)

print("Key loaded:", bool(os.environ.get("OPENAI_API_KEY")))

Key loaded: True


---
## Step 4: Understand the Use Case — Product Description Writer

### 🛒 Real-World Problem
You work at an e-commerce company (like Flipkart, Meesho, or Amazon India).  
Every day, hundreds of new products are added to the platform.  
Writing a good product description for each one **takes time and effort**.

### ✅ Our Solution
We build a tool where a seller just provides:
- **Product Name** (e.g., "Wireless Bluetooth Earbuds")
- **Key Features** (e.g., "30-hour battery, noise cancellation, IPX5 waterproof")
- **Target Audience** (e.g., "college students and gym-goers")

And GPT automatically writes a **compelling, professional product description**.

### 🏷️ Sample Products We Will Test With

| # | Product | Features |
|---|---|---|
| 1 | Wireless Bluetooth Earbuds | 30hr battery, ANC, IPX5 waterproof, fast charge |
| 2 | Yoga Mat | 6mm thick, non-slip, eco-friendly, carry strap |
| 3 | Stainless Steel Water Bottle | 1L, keeps cold 24hr, BPA free, leak-proof lid |

We will start with Product 1 and then you can try the others.


---
## Step 5: Create the Prompt Template

### 📝 What Is a Prompt?
A **prompt** is the instruction you give to GPT — just like telling a human writer what to do.

The quality of your output depends heavily on the quality of your prompt.  
This is called **Prompt Engineering**.

### 📋 What Is a Prompt Template?
A **Prompt Template** is a prompt with **placeholders** (variables).  
Instead of writing a new prompt every time, you define it once with `{variable_name}` slots,  
and fill in the actual values when you run it.

### 🔍 Breaking Down Our Template:

```
system  → Tells GPT what ROLE to play ("You are an expert e-commerce copywriter")
human   → The actual TASK with variable slots: {product_name}, {features}, {audience}
```

The **system message** sets the context (like a job description for the AI).  
The **human message** is the actual request (like an email you send to the expert).


In [4]:
# Define the Prompt Template with placeholders
prompt_template = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are an expert e-commerce copywriter with 10 years of experience 
writing compelling product descriptions for online shopping platforms. 
Your descriptions are clear, engaging, benefit-focused, and persuasive.
Always write in a friendly yet professional tone."""
    ),
    (
        "human",
        """Write a product description for an e-commerce listing.

Product Name: {product_name}
Key Features: {features}  
Target Audience: {target_audience}

Requirements:
- Start with a catchy one-line hook
- Highlight 3-4 key benefits (not just features — explain WHY each matters)
- End with a short call-to-action
- Keep the total length between 80-120 words
- Use simple, everyday language"""
    )
])

print("✅ Prompt Template created successfully!")
print()
print("📋 Template Preview:")
print("   System Role : Expert e-commerce copywriter")
print("   Input Variables : {product_name}, {features}, {target_audience}")
print("   Output Format : Hook → Benefits → Call-to-action")

✅ Prompt Template created successfully!

📋 Template Preview:
   System Role : Expert e-commerce copywriter
   Input Variables : {product_name}, {features}, {target_audience}
   Output Format : Hook → Benefits → Call-to-action


---
## Step 6: Initialize the GPT Model

### 🤖 What Is `ChatOpenAI`?
`ChatOpenAI` is LangChain's class that **represents** the GPT model.  
When you create this object, you are configuring *which* model to use and *how* it should behave.

### ⚙️ Key Parameters Explained:

| Parameter | Value | What It Means |
|---|---|---|
| `model` | `"gpt-3.5-turbo"` | Which GPT version to use. 3.5-turbo is fast and cost-effective |
| `temperature` | `0.7` | Controls **creativity** of the output |

### 🌡️ Understanding Temperature:

```
temperature = 0.0  →  Very predictable, factual, repetitive
temperature = 0.7  →  Balanced — creative but still on-topic  ✅ (our choice)
temperature = 1.0  →  Very creative, sometimes unpredictable
temperature = 2.0  →  Very random, often goes off-topic
```

For product descriptions, **0.7 is the sweet spot** — creative enough to be engaging,  
but focused enough to stay relevant to the product.


In [5]:
# Initialize the GPT model via LangChain
llm = ChatOpenAI(
    model="gpt-3.5-turbo",   # The GPT model version to use
    temperature=0.7           # Creativity level: 0 = robotic, 1 = very creative
)

print("✅ GPT Model initialized!")
print(f"   Model    : gpt-3.5-turbo")
print(f"   Temperature : 0.7  (balanced creativity)")

✅ GPT Model initialized!
   Model    : gpt-3.5-turbo
   Temperature : 0.7  (balanced creativity)


---
## Step 7: Build the Chain — The Heart of LangChain

### 🔗 What Is a "Chain"?
This is where LangChain gets its name.

A **Chain** is a sequence of steps connected together using the `|` (pipe) operator.  
Each step's **output** becomes the next step's **input**.

### 🏭 Our Chain Has 3 Steps:

```
prompt_template  →  llm  →  output_parser

Step 1: prompt_template  
        Takes your inputs (product name, features, audience)  
        Fills them into the template  
        Produces a formatted message ready for GPT  

Step 2: llm (ChatOpenAI)  
        Receives the formatted prompt  
        Sends it to OpenAI's GPT  
        Gets back the AI's response object  

Step 3: output_parser (StrOutputParser)  
        Takes the response object from GPT  
        Extracts just the text content  
        Returns a clean Python string  
```

### 💡 Why Use a Chain?
Without a chain, you would have to call each step manually and pass results around.  
With a chain, you call `.invoke()` once and LangChain handles everything automatically.


In [6]:
# Initialize the output parser
output_parser = StrOutputParser()

# Build the chain by connecting the 3 steps with | (pipe operator)
chain = prompt_template | llm | output_parser

#  chain reads as:
#  "Take the prompt template → send to GPT → parse the output"

print("✅ Chain built successfully!")
print()
print("🔗 Chain Flow:")
print("   [Prompt Template]  →  [GPT Model]  →  [Output Parser]")
print()
print("   prompt_template  |  llm  |  output_parser")

✅ Chain built successfully!

🔗 Chain Flow:
   [Prompt Template]  →  [GPT Model]  →  [Output Parser]

   prompt_template  |  llm  |  output_parser


---
## Step 8: Generate a Product Description — Wireless Earbuds

### 🚀 How `.invoke()` Works

```python
chain.invoke({ "variable_name": "value" })
```

You pass a **dictionary** with keys matching the template's placeholders.  
LangChain fills in the template, calls GPT, parses the result, and returns the final text.

### 🎧 Product 1: Wireless Bluetooth Earbuds
We will generate a description for a popular product category.

Watch what happens when you run the cell below:


In [7]:
# Product 1: Wireless Bluetooth Earbuds
product_input = {
    "product_name"    : "ProSound X1 Wireless Bluetooth Earbuds",
    "features"        : "30-hour total battery life, Active Noise Cancellation (ANC), IPX5 waterproof, 10-minute fast charge gives 2 hours playback, Bluetooth 5.3",
    "target_audience" : "college students, gym-goers, and daily commuters who want premium audio without the premium price"
}

print("⏳ Sending request to GPT... please wait")
print("-" * 55)

# Invoke the chain — this sends the prompt to OpenAI and returns the result
description = chain.invoke(product_input)

print("✅ Product Description Generated!")
print("=" * 55)
print()
print(description)
print()
print("=" * 55)

⏳ Sending request to GPT... please wait
-------------------------------------------------------
✅ Product Description Generated!

Experience premium sound without breaking the bank with our ProSound X1 Wireless Bluetooth Earbuds. Enjoy uninterrupted music for up to 30 hours, thanks to the long-lasting battery life and Active Noise Cancellation, perfect for drowning out distractions during study sessions or workouts. Don't let sweat or rain stop your groove with the IPX5 waterproof rating. Plus, a quick 10-minute charge gives you an extra 2 hours of playtime, ensuring your music never skips a beat. Upgrade your audio game today!



---
## Step 9: Test With a Different Product — Yoga Mat

### 🔁 Reusability of the Chain
Notice that we do NOT need to rebuild the chain.  
We just call `.invoke()` again with **different input values**.

This is the power of Prompt Templates + Chains —  
**one setup, many different outputs.**

### 🧘 Product 2: Yoga Mat


In [8]:
# Product 2: Yoga Mat
product_input_2 = {
    "product_name"    : "FlexFit Pro Yoga Mat",
    "features"        : "6mm thick cushioning, double-sided non-slip texture, eco-friendly TPE material, free carry strap, available in 5 colors",
    "target_audience" : "yoga beginners and fitness enthusiasts who practice at home or in the gym"
}

print("⏳ Sending request to GPT... please wait")
print("-" * 55)

description_2 = chain.invoke(product_input_2)

print("✅ Product Description Generated!")
print("=" * 55)
print()
print(description_2)
print()
print("=" * 55)

⏳ Sending request to GPT... please wait
-------------------------------------------------------
✅ Product Description Generated!

Elevate your yoga practice with the FlexFit Pro Yoga Mat! Unwind in comfort with its 6mm thick cushioning, providing joint support and stability for every pose. The double-sided non-slip texture keeps you grounded, ensuring a safe and focused practice. Crafted from eco-friendly TPE material, this mat is gentle on the planet and your body. Plus, enjoy the convenience of a free carry strap for easy transport to the studio or park. Available in 5 vibrant colors to suit your style. Embrace your practice with confidence — get your FlexFit Pro Yoga Mat today!



---
## Step 10: 🧪 Your Turn — Try Your Own Product!

Now it is your turn to test the system with a product of your choice.

### Instructions:
1. Replace the values in the dictionary below with your own product details
2. Run the cell
3. See GPT generate a description for it!

### 💡 Ideas to Try:
- A smartphone
- A kitchen appliance (mixer, air fryer)
- A study desk or chair
- A skincare product

Go ahead — make it yours! 👇


In [9]:
# ✏️ YOUR TURN — Replace these values with your own product
my_product = {
    "product_name"    : "Toddler Tri cycle",
    "features"        : "three wheels, handles for pushing, easy to pedale",
    "target_audience" : "Todler parents"
}

print("⏳ Sending your product to GPT...")
print("-" * 55)

my_description = chain.invoke(my_product)

print("✅ Your Product Description:")
print("=" * 55)
print()
print(my_description)
print()
print("=" * 55)

⏳ Sending your product to GPT...
-------------------------------------------------------
✅ Your Product Description:

Introducing the Toddler Tri-cycle: the perfect ride for your little explorer! With its three sturdy wheels, convenient handles for easy pushing, and pedals that are a breeze for tiny feet to use, this tri-cycle is designed to make your toddler's playtime both fun and safe. Watch as your child develops balance and coordination while having a blast outdoors. Give your little one the gift of endless adventures with this adorable tri-cycle. Order now and let the fun begin!



---
## Step 11: 🌡️ Experiment — How Does Temperature Change the Output?

### What We Are Doing Here
We will generate descriptions for the **same product** using **different temperatures**  
so you can directly see how the creativity level affects the writing style.

### What to Observe:
- **Low temperature (0.1)** → Safe, predictable, to-the-point writing
- **High temperature (0.9)** → More expressive, varied, sometimes surprising


In [10]:
# Experiment: Same product, two different temperatures
test_product = {
    "product_name"    : "AquaCool Stainless Steel Water Bottle",
    "features"        : "1 litre capacity, keeps drinks cold for 24 hours, BPA-free, leak-proof lid, fits standard cup holders",
    "target_audience" : "office workers, travellers, and fitness enthusiasts"
}

temperatures = [0.1, 0.9]

for temp in temperatures:
    print(f"{'='*55}")
    print(f"🌡️  Temperature = {temp}  ({'Conservative' if temp < 0.5 else 'Creative'})")
    print(f"{'='*55}")
    
    # Create a new model instance with this temperature
    temp_llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=temp)
    temp_chain = prompt_template | temp_llm | output_parser
    
    result = temp_chain.invoke(test_product)
    print(result)

print(f"{'='*55}")
print("💡 Notice the difference in tone and word choice between the two!")
print(f"{'='*55}")

🌡️  Temperature = 0.1  (Conservative)
Stay cool and hydrated on-the-go with the AquaCool Stainless Steel Water Bottle! With a generous 1-liter capacity, this bottle is a must-have for office workers, travellers, and fitness enthusiasts. Keep your drinks icy cold for a refreshing 24 hours, thanks to its double-wall insulation. Worried about harmful chemicals? Fear not, as this BPA-free bottle ensures safe sipping. Plus, its leak-proof lid and convenient size make it a perfect fit for standard cup holders. Stay refreshed, stay healthy — grab your AquaCool bottle today!
🌡️  Temperature = 0.9  (Creative)
Stay cool on the go with the AquaCool Stainless Steel Water Bottle! Say goodbye to lukewarm sips with its impressive 24-hour cold retention. Made for those always on the move, this BPA-free bottle fits in your cup holder and boasts a leak-proof lid for mess-free hydration. Perfect for office warriors, jet-setters, and gym buffs who deserve a refreshingly chilled drink anytime, anywhere. St

---
## ✅ Lab Complete — What Did You Build?

Congratulations! You have successfully built an **AI-powered Product Description Generator** using:
- **LangChain** → to structure and manage the AI workflow
- **OpenAI GPT** → the pre-trained language model that generates the text
- **Prompt Templates** → reusable, variable-driven instructions for the AI
- **Chains** → a clean pipeline that connects all steps automatically

---

## 📚 Concepts Recap

| Concept | What It Is | Analogy |
|---|---|---|
| **Pre-trained Model** | GPT already learned from the internet — we just use it | Hiring an expert writer |
| **Prompt** | The instruction you give to GPT | A task description you send by email |
| **Prompt Template** | A reusable prompt with variable slots | A form letter with blank fields |
| **Temperature** | Controls how creative or predictable the output is | Adjusting a creativity dial |
| **Chain** | A connected pipeline of steps | An assembly line in a factory |
| **`.invoke()`** | Runs the entire chain with your input | Pressing the "Generate" button |

---

## 🚀 What Can You Build Next?

With the same approach, you can build:
- 📧 **Customer support email generator**
- 📰 **News article summarizer**
- 🏦 **Loan approval/rejection letter writer**
- 🩺 **Patient discharge summary generator**
- 📝 **Resume bullet point writer**

The **only thing that changes** is the Prompt Template — the LangChain pattern stays the same!

---

*Lab: Generate Text with Pre-trained GPT Model | LangChain + OpenAI | E-commerce Use Case*
